## First, make standardized matrix files. 
- largely similar to the 06_dataquality, but slightly different because the reference is not consistent. 

In [ ]:
import pybedtools, os, re
import pandas as pd
import numpy as np
from pathlib import Path

def FindMissing(data_dir, out_dir, prefix):
    in_files = [x for x in os.listdir(data_dir) if x.endswith(prefix) ]  
    out_files = [x for x in os.listdir(out_dir) if x.endswith(".mat") ]
    out_names = [re.sub(".mat", prefix, f_foot, count=1) for f_foot in out_files]
    left = list(set(in_files).difference(set(out_names)))
    return left


all_dirs = ["01_original/union/", "05_cellsim"] 

tool = "hint"
for dd in all_dirs:
    out_dir = f"../07_cellsim/{tool}/"
    data_dir = f"../05_footprinting/{dd}/{tool}"
    all_files  = FindMissing(data_dir, f"{out_dir}/mats/", "_mpbs.bed")
    if len(all_files) != 0 : print(f"Missing the mat files for {tool}:{dd}\n{all_files}")


tool = "tobias"
for dd in all_dirs:
    out_dir = f"../07_cellsim/{tool}/"
    data_dir = f"../05_footprinting/{dd}/{tool}"
    all_files  = FindMissing(data_dir, f"{out_dir}/mats/", "_bindetect_TF_overviews.txt")
    if len(all_files) != 0 : print(f"Missing the mat files for {tool}:{dd}\n{all_files}")


tool = "print"
for dd in all_dirs:
    out_dir = f"../07_cellsim/{tool}/"
    data_dir = f"../05_footprinting/{dd}/{tool}"
    all_files  = FindMissing(data_dir, f"{out_dir}/mats/", "_granges.bed")
    if len(all_files) != 0 : print(f"Missing the mat files for {tool}:{dd}\n{all_files}")

print("done")

### HINT

In [ ]:
###   HINT  ###
import pybedtools, os, re
import pandas as pd
import numpy as np
from pathlib import Path

out_dir = "../07_cellsim/hint/"
data_dir = "../05_footprinting/05_cellsim/hint"

output_cleaned_beds = True
if output_cleaned_beds: Path(f"{out_dir}/cleaned/").mkdir(parents=True, exist_ok=True)
Path(f"{out_dir}/mats/").mkdir(parents=True, exist_ok=True)
    
all_files = [x for x in os.listdir(data_dir) if x.endswith("_mpbs.bed") ]
all_files  = FindMissing(data_dir, f"{out_dir}/mats/", "_mpbs.bed")

#get all the atac_regions
peak_file = f"../03_peakcalls/Union_filt_500bp.exclusion.bed.gz"
df = pd.read_csv(f"{peak_file}", header=None, sep="\t")
df["region_atac"] = df[0].astype(str) + ":" + df[1].astype(str) + "-" + df[2].astype(str)
all_atac_regions = df["region_atac"].tolist()

#get all the tfs and add a way to standardize the names. 
infile="../05_footprinting/program_input_files/jaspar2022_motifs_to_clusters.txt.gz"
TF_df = pd.read_csv(infile, sep=" ", header=None, usecols = [0], names=["motif"])
TF_df["motif"] = [i.upper() for i in TF_df["motif"]] 
TF_df["hint_convert"] = [i.split('_')[1]+"."+i.split('_')[0] for i in TF_df["motif"]]
all_tfs = TF_df["motif"].tolist()

#for all files and threshold combos, make the matrices. 
for f_foot in all_files:
    name = re.sub('_mpbs.bed', '', f_foot, count=1)
    print(f"working on: {name}")

    #interesct the two files so we have the tag counts. 
    file2 = re.sub('_mpbs.bed', '.bed', f_foot, count=1)
    ! bedtools intersect \
        -a <(awk '{{print $$1,$$2,$$3,$$5}}' {data_dir}/{file2} | tr " " "\t") \
        -b <(awk '{{sub(/[ \t]+$$/, ""); print}}' {data_dir}/{f_foot} | sort -k1,1 -k2,2n) \
        -F 1 -wb -wa -sorted | awk '{{print $$5,$$6,$$7,$$8,$$9,$$4}}' > current_hint.bed

    #intersect again with our peak file.
    #note the 50% here is because hint ft can overhang the peak regions. 
    #interestingly there are a few that fall very near but technically outside. it must be 
    #part of how the original bed files are constructed. 
    #so the numbers between mpbs files and these beds will be slightly off. 
    ! bedtools intersect \
        -a {peak_file} \
        -b <(cat current_hint.bed | tr " " "\t") \
        -F 0.5 -wb -wa -sorted > current_hint2.bed

    #read in df and merge by tag count. 
    df = pd.read_csv("current_hint2.bed", sep="\t", header=None)
    df = df.drop_duplicates()
    df["region_atac"] = df[0].astype(str) + ":" + df[1].astype(str) + "-" + df[2].astype(str)
    df["region_TFBS"] = df[6].astype(str) + ":" + df[7].astype(str) + "-" + df[8].astype(str)
    df["curr_motif"] = [i.upper() for i in df[9]]
    df["pwm_score"] = df[10]
    df["tag_count"] = df[11]
    df1 = df.merge(TF_df, right_on='hint_convert', left_on="curr_motif", how='left')

    if output_cleaned_beds:
        df1[["region_atac", "region_TFBS", "motif","pwm_score","tag_count"]].to_csv(f"{out_dir}/cleaned/{name}.bed", sep="\t", index=False)

    df_observed = df1.pivot_table(index="region_atac", columns="motif", aggfunc='size', fill_value=0)
    df_observed = df_observed.reindex(all_tfs, axis=1, fill_value=0)
    df_observed = df_observed.reindex(all_atac_regions, axis=0, fill_value=0)

    if df.shape[0] == df_observed.sum().sum():
        df_observed.to_csv(f"{out_dir}/mats/{name}.mat", index=True)
    else:
        print("ERRORRRRR")


### TOBIAS

In [ ]:
import pybedtools, os, re
import pandas as pd
import numpy as np
from pathlib import Path

out_dir = "../07_cellsim/tobias/"
data_dir = "../05_footprinting/05_cellsim/tobias"

output_cleaned_beds = False
if output_cleaned_beds: Path(f"{out_dir}/cleaned/").mkdir(parents=True, exist_ok=True)
Path(f"{out_dir}/mats/").mkdir(parents=True, exist_ok=True)

all_files = [x for x in os.listdir(data_dir) if x.endswith("_bindetect_TF_overviews.txt") ]
all_files  = FindMissing(data_dir, f"{out_dir}/mats/", "_bindetect_TF_overviews.txt")

#get all the atac_regions
peak_file = f"../03_peakcalls/Union_filt_500bp.exclusion.bed.gz"
df = pd.read_csv(f"{peak_file}", header=None, sep="\t")
df["region_atac"] = df[0].astype(str) + ":" + df[1].astype(str) + "-" + df[2].astype(str)
all_atac_regions = df["region_atac"].tolist()

#get all the tfs and add a way to standardize the names. 
infile="../05_footprinting/program_input_files/jaspar2022_motifs_to_clusters.txt.gz"
TF_df = pd.read_csv(infile, sep=" ", header=None, usecols = [0], names=["motif"])
TF_df["motif"] = [i.upper() for i in TF_df["motif"]] 
TF_df["tobias_convert"] = [re.sub('::', '', i) for i in TF_df["motif"]]
all_tfs = TF_df["motif"].tolist()

#for all files and threshold combos, make the matrices. 
for f_foot in all_files:
    name = re.sub('_bindetect_TF_overviews.txt', '', f_foot, count=1)
    print(f"working on: {name}")

    #make a file we can work with
    #cmd = f'''awk '$14==1 {{split($4, a, "[_]"); $4 = a[1]"_"a[2]; print $7":"$8"-"$9, $1":"$2"-"$3, toupper($4), $13}}' {data_dir}/{f_foot} > current_tobias.bed'''
    cmd = f'''awk '$13 > 0.025 {{split($4, a, "[_]"); $4 = a[1]"_"a[2]; print $7":"$8"-"$9, $1":"$2"-"$3, toupper($4), $13}}' {data_dir}/{f_foot} > current_tobias.bed'''
    ! {cmd}

    df = pd.read_csv("current_tobias.bed", sep=" ", names=["region_atac", "region_TFBS", "curr_motif", "score"])
    df1 = df.merge(TF_df, right_on='tobias_convert', left_on="curr_motif", how='left')

    if output_cleaned_beds:
        df1[["region_atac", "region_TFBS", "motif","score"]].to_csv(f"{out_dir}/cleaned/{name}.bed", sep="\t", index=False)

    df_observed = df1.pivot_table(index="region_atac", columns="motif", aggfunc='size', fill_value=0)
    df_observed = df_observed.reindex(all_tfs, axis=1, fill_value=0)
    df_observed = df_observed.reindex(all_atac_regions, axis=0, fill_value=0)

    if df.shape[0] == df_observed.sum().sum():
        df_observed.to_csv(f"{out_dir}/mats/{name}.mat", index=True)
    else:
        print("ERRORRRRR")

### PRINT

In [ ]:
import pybedtools, os, re
import pandas as pd
import numpy as np
from pathlib import Path

out_dir = "../07_cellsim/print/"
data_dir = "../05_footprinting/05_cellsim/print/"

output_cleaned_beds = False
if output_cleaned_beds: Path(f"{out_dir}/cleaned/").mkdir(parents=True, exist_ok=True)
Path(f"{out_dir}/mats/").mkdir(parents=True, exist_ok=True)

all_files = [x for x in os.listdir(data_dir) if x.endswith("_granges.bed") ]
all_files  = FindMissing(data_dir, f"{out_dir}/mats/", "_granges.bed")

#get all the atac_regions
peak_file = f"../03_peakcalls/Union_filt_500bp.exclusion.bed.gz"
df = pd.read_csv(f"{peak_file}", header=None, sep="\t")
df["region_atac"] = df[0].astype(str) + ":" + df[1].astype(str) + "-" + df[2].astype(str)
all_atac_regions = df["region_atac"].tolist()

#get all the tfs and add a way to standardize the names. 
infile="../05_footprinting/program_input_files/jaspar2022_motifs_to_clusters.txt.gz"
TF_df = pd.read_csv(infile, sep=" ", header=None, usecols = [0], names=["motif"])
TF_df["motif"] = [i.upper() for i in TF_df["motif"]] 
all_tfs = TF_df["motif"].tolist()

for f_foot in all_files:
    name = re.sub('_granges.bed', '', f_foot, count=1)
    #name = re.sub('_granges.bed', '-cellFilt', f_foot, count=1)
    print(f"working on: {name}")

    #intersect with our peak file.
    #this is really overkill since all the files are in the same order, but leaving it for consistencies sake. 
    ! bedtools intersect \
        -a {peak_file} \
        -b <(tail -n+2 {data_dir}{f_foot} | tr " " "\t" | sort -k1,1 -k2,2n) \
        -F 0.5 -wb -wa -sorted > current_print.bed

    #read in df and merge by tag count. 
    df = pd.read_csv("current_print.bed", sep="\t", header=None) 
    df = df.drop_duplicates()
    df["region_atac"] = df[0].astype(str) + ":" + df[1].astype(str) + "-" + df[2].astype(str)
    df["region_TFBS"] = df[6].astype(str) + ":" + df[7].astype(str) + "-" + df[8].astype(str)
    df["motif"] = [i.upper() for i in df[12]]
    df["score"] = df[13]
    
    if output_cleaned_beds:
        df[["region_atac", "region_TFBS", "motif","score"]].to_csv(f"{out_dir}/cleaned/{name}.bed", sep="\t", index=False)
            
    df = df[df["score"] >= 0.3]
    df_observed = df.pivot_table(index="region_atac", columns="motif", aggfunc='size', fill_value=0)
    df_observed = df_observed.reindex(all_tfs, axis=1, fill_value=0)
    df_observed = df_observed.reindex(all_atac_regions, axis=0, fill_value=0)
    
    if df.shape[0] == df_observed.sum().sum():
        df_observed.to_csv(f"{out_dir}/mats/{name}.mat", index=True)
    else:
        print("ERRORRRRR")

## Next, calculate metrics 

In [ ]:
import os, re, pickle
import pandas as pd
import numpy as np
from itertools import chain

def _remove_strings(text):
    #remove common strings from test. 
    patterns = [r'\.summary\.txt', r'\.mat', r'-cellFilt', r'\_metrics.csv']
    for pattern in patterns:
        text = re.sub(pattern, '', text)
    return text


def _make_contingency_table(df_expected, df_observed): 
    contingency_table = {
            'TP': np.sum(np.minimum(df_observed, df_expected), axis=0),      # Minimum of the two values is true positive
            'FP': np.sum(np.maximum(0, df_observed - df_expected), axis=0),  # Difference, where df_observed > df_expected
            'FN': np.sum(np.maximum(0, df_expected - df_observed), axis=0),  # Difference, where df_expected > df_observed
            'TN': np.sum((df_observed == 0) & (df_expected == 0), axis=0)
        }
    contingency_df = pd.DataFrame(contingency_table)
    return(contingency_df)

    
    
def _calc_scores(df, averaging_type):
    
    if averaging_type=="micro":
        #micro = sum all TPs and then calculate F1
        #will return three values, 
        
        tp = df["TP"].sum()
        fp = df["FP"].sum()
        fn = df["FN"].sum()
        if tp==0 and fp==0 and fn==0: 
            precision = 0; recall=0; f1=0
        else:  
            precision = (tp/(tp+fp))
            recall = (tp/ (tp+fn))
            f1 = ( tp/ (tp + (0.5*(fp+fn)) ) )
            
        return pd.Series([precision, recall, f1], index=["precision","recall","f1"])

    elif averaging_type=="macro":
        #macro = calculate F1s and then take average F1. 
        #will return three lists, 
        
        df['precision'] = np.where(df['TP'] + df['FP'] == 0, np.nan, 
                                   df['TP'] / (df['TP'] + df['FP']))
        df['recall']    = np.where(df['TP'] + df['FN'] == 0, np.nan, 
                                   df['TP'] / (df['TP'] + df['FN']))
        df['f1']        = np.where(df['TP'] + (0.5 * (df['FP'] + df['FN'])) == 0, np.nan,
                                   df['TP'] / (df['TP'] + (0.5 * (df['FP'] + df['FN']))))
        return(df)
        
        
        
    
def CalculateMetrics(input_file, mat_dir, cell_line, keep_peaks=None, analysis_type="tf"):

    df_expected = pd.read_csv(f"{mat_dir}/{cell_line}-cellFilt.mat", index_col=0)
    df_observed = pd.read_csv(f"{mat_dir}/{input_file}", index_col=0)
    
    if keep_peaks is not None:  
        df_expected = df_expected.loc[keep_peaks]
        df_observed = df_observed.loc[keep_peaks] 

    #read in observed
    cond_name = f"{re.sub('.mat','', input_file)}_{cell_line}"
    print(f"...{cond_name}")

    if analysis_type == "peak":
        df_expected_peak = df_expected.sum(axis=1).to_frame(name='sum')
        df_observed_peak = df_observed.sum(axis=1).to_frame(name='sum')
        cont_df = _make_contingency_table(df_expected_peak, df_observed_peak)

    elif analysis_type == "tf":
        cont_df = _make_contingency_table(df_expected, df_observed)

    else:
        print("internal error in calculte metrics"); return


    cont_df_with_metrics = _calc_scores(cont_df, averaging_type="macro")
    cont_df_with_metrics["type"] = analysis_type

    outfile = f"{metric_dir}/{cond_name}_metrics.csv"
    if not os.path.isfile(outfile):
        cont_df_with_metrics.to_csv(outfile)
    else: 
        print("appending results to existing file") #so you can stack tf, cluster, peak, etc. 
        cont_df_with_metrics.to_csv(outfile, mode='a', header=False)

    return
    
    
    
def JoinMetrics(metric_dir, analysis_type):
    files = [x for x in os.listdir(metric_dir) if x.endswith("_metrics.csv")]
    
    joint = pd.DataFrame()
    for infile in files:
        cond_name = _remove_strings(infile)

        byTF_df = pd.read_csv(f"{metric_dir}/{infile}", index_col=0)
        byTF_df = byTF_df.loc[byTF_df["type"] == analysis_type]

        micro_df = _calc_scores(byTF_df, averaging_type="micro")
        mean_df = byTF_df[["precision", "recall", "f1"]].apply(lambda col: col.mean())
        std_df  = byTF_df[["precision", "recall", "f1"]].apply(lambda col: col.std())
        joint[cond_name] = pd.concat([micro_df.add_suffix('_micro'),
                                      mean_df.add_suffix('_mean'), 
                                      std_df.add_suffix('_sd')], axis=0)
    
    return joint.T

### Baseline version

In [ ]:
import os, re, pickle
import pandas as pd
import numpy as np
from pathlib import Path

t = "print" 
data_dir = f"../07_cellsim/{t}/" #will look for mats
mat_dir = f"{data_dir}/mats/"
metric_dir = f"{data_dir}/metrics/"
Path(metric_dir).mkdir(parents=True, exist_ok=True)

files = [x for x in os.listdir(mat_dir) if x.startswith("combo")]
has_metrics = [re.sub('_metrics.csv','', x) for x in os.listdir(metric_dir) if x.startswith("combo")]

#read in metadata so we know what cell line is the reference. 
all_metrics = pd.read_csv("../04_downsampling/05_cellsim/sampling_stats_cellsim.csv", sep=",")
all_metrics['filename'] = all_metrics['filename'].str.replace(".bam", '', regex=False)
all_metrics.index = all_metrics['filename'] + "_" + all_metrics['dist_to_*']

for f in files:
    name = re.sub('.mat','', f)
    tmp = all_metrics.loc[all_metrics["filename"] == name] 
    
    #all reference cell lines for this file, can be more than one though usually not. 
    for cl in tmp["dist_to_*"].values: 
        cond_name = f"{re.sub('.mat','', f)}_{cl}"
        if cond_name in has_metrics: continue
        
        CalculateMetrics(input_file = f,
                         mat_dir = mat_dir,
                         cell_line = cl, 
                         analysis_type = "tf") 
        
        
df_file = JoinMetrics(metric_dir, "tf")
df_INFO = df_file.merge(all_metrics, right_index=True, left_index=True, how='left')
outfile=f"{metric_dir}/{t}_bytf.csv"
df_INFO.to_csv(outfile)

### Random TF subset

In [ ]:
import os, re, pickle, random
import pandas as pd
import numpy as np
from pathlib import Path

f = "../07_cellsim/print/metrics/combo_0_HEPG2_metrics.csv"
tmp = pd.read_csv(f"{f}", sep=",", index_col=0)
processed_tfs = random.sample(list(tmp.dropna(subset=['f1']).index),  350)


#read in metadata so we know what cell line is the reference. 
all_metrics = pd.read_csv("../04_downsampling/05_cellsim/sampling_stats_cellsim.csv", sep=",")
all_metrics['filename'] = all_metrics['filename'].str.replace(".bam", '', regex=False)
all_metrics.index = all_metrics['filename'] + "_" + all_metrics['dist_to_*']

analysis_type = "tf"

for t in ["hint", "tobias", "print"]:
    data_dir = f"../07_cellsim/{t}/" #will look for mats
    metric_dir = f"{data_dir}/metrics_100/"
    files = [x for x in os.listdir(metric_dir) if x.endswith("_metrics.csv")]
    
    joint = pd.DataFrame()
    for infile in files:
        cond_name = _remove_strings(infile)

        byTF_df = pd.read_csv(f"{metric_dir}/{infile}", index_col=0)
        byTF_df = byTF_df.loc[byTF_df["type"] == analysis_type]
        byTF_df = byTF_df.loc[processed_tfs]
        
        micro_df = _calc_scores(byTF_df, averaging_type="micro")
        mean_df = byTF_df[["precision", "recall", "f1"]].apply(lambda col: col.mean())
        std_df  = byTF_df[["precision", "recall", "f1"]].apply(lambda col: col.std())
        joint[cond_name] = pd.concat([micro_df.add_suffix('_micro'),
                                      mean_df.add_suffix('_mean'), 
                                      std_df.add_suffix('_sd')], axis=0)

    df_file = joint.T
    df_INFO = df_file.merge(all_metrics, right_index=True, left_index=True, how='left')
    outfile=f"{metric_dir}/{t}_bytf_limitedTFset.csv"
    df_INFO.to_csv(outfile)

### Limited Peak Set

In [ ]:
import multiprocessing as mp
from functools import partial
import pyBigWig

def mean_cov(row, bw):
    coverage = bw.stats(row["chr"], row["start"], row["stop"], type="mean")
    return coverage[0] if coverage else 0

def process_chunk(f, peak_df):
    print(f)
    with pyBigWig.open(f) as bw:
        res = peak_df.apply(lambda row: mean_cov(row, bw), axis=1)  # Ensure bw is passed correctly
    return res


def main():
    data_dir = "../07_cellsim/tfbs_universe/"

    peak_df=pd.read_csv(f"../03_peakcalls/Union_filt_500bp.exclusion.bed.gz", sep="\t", 
                header=None, index_col = False, 
                names=["chr","start","stop","peakname", "summit_score", "strand"])
    peak_df["region_atac"] = peak_df["chr"].astype(str) + ":" + peak_df["start"].astype(str) + "-" + peak_df["stop"].astype(str)

    files = [f"{data_dir}/{x}" for x in os.listdir(data_dir)]

    with mp.Pool(processes=10) as pool:
        results = pool.map(partial(process_chunk, peak_df=peak_df), files)

    df = pd.concat(results, axis=1)
    df.columns = [re.sub(".bw" , "", os.path.basename(x)) for x in files]
    df["summit_score"] = peak_df["summit_score"]
    df.index = peak_df["region_atac"]
    
    df.to_csv(f"../07_cellsim/tfbs_universe/PeakCoverage.csv")
    return df

#main()
### dont rerun! ###

In [ ]:
import os, re, pickle
import pandas as pd
import numpy as np
from pathlib import Path

t = "hint" 
peak_threshold = 100

data_dir = f"../07_cellsim/{t}/" #will look for mats
mat_dir = f"{data_dir}/mats/"
metric_dir = f"{data_dir}/metrics_{peak_threshold}/"
Path(metric_dir).mkdir(parents=True, exist_ok=True)

#get all files that are missing metric files for this section
files = [x for x in os.listdir(mat_dir) if x.startswith("combo")]
has_metrics = [re.sub('_metrics.csv','', x) for x in os.listdir(metric_dir) if x.startswith("combo")]

#read in metadata so we know what cell line is the reference. 
all_metrics = pd.read_csv("../04_downsampling/05_cellsim/sampling_stats_cellsim.csv", sep=",")
#all_metrics["filename"] = [i.replace('.bam', '.mat') for i in all_metrics["filename"].values]
all_metrics['filename'] = all_metrics['filename'].str.replace(".bam", '', regex=False)
all_metrics.index = all_metrics['filename'] + "_" + all_metrics['dist_to_*']

peak_df = pd.read_csv(f"../07_cellsim/tfbs_universe/PeakCoverage.csv.gz", index_col=0)

for f in files:
    name = re.sub('.mat','', f)
    tmp = all_metrics.loc[all_metrics["filename"] == name] 
    
    #all reference cell lines for this file, can be more than one though usually not. 
    for cl in tmp["dist_to_*"].values: 
        cond_name = f"{name}_{cl}"
        if cond_name in has_metrics: continue
        
        CalculateMetrics(input_file = f,
                         mat_dir = mat_dir,
                         cell_line = cl, 
                         analysis_type = "tf", 
                         keep_peaks = peak_df[(peak_df[name] > peak_threshold)].index) 
        
        
df_file = JoinMetrics(metric_dir, "tf")
df_INFO = df_file.merge(all_metrics, right_index=True, left_index=True, how='left')
outfile=f"{metric_dir}/{t}_bytf.csv"
df_INFO.to_csv(outfile)